# 附：Codes .{unnumbered}

本 Notebook 用于生成 `06_heckman_lec_v2.ipynb` 中调用的企业贷款选择与贷款利率模拟数据，以及相关图片。

In [ ]:
# ============================================================
# 0. 全局设置
# ============================================================

import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from scipy import stats
import statsmodels.api as sm

FIG_DIR = Path("./figs")
DATA_DIR = Path("./data")
FIG_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

font_paths = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/opentype/noto/NotoSerifCJK-Regular.ttc",
    "C:/Windows/Fonts/simhei.ttf",
    "C:/Windows/Fonts/msyh.ttc",
]

for fp in font_paths:
    if os.path.exists(fp):
        fm.fontManager.addfont(fp)
        font_prop = fm.FontProperties(fname=fp)
        plt.rcParams["font.family"] = font_prop.get_name()
        break

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams.update({
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=300)
    fig.savefig(FIG_DIR / f"{name}.svg", bbox_inches="tight")
    plt.close(fig)

In [ ]:
# ============================================================
# 1. 生成 Heckman 选择模型模拟数据
# ============================================================

def simulate_heckman_credit(n=3200, seed=20260501):
    """
    生成 Heckman 章节使用的数据。

    loan_rate 只在获得贷款的企业中可观测。
    选择方程误差 v 与利率方程误差 u 相关，因此 naive OLS 会受到选择性观测影响。
    """
    r = np.random.default_rng(seed)

    collateral = (r.beta(2.2, 2.0, n) - 0.52) / 0.22
    bank_access = r.normal(0, 1, n)
    risk = r.normal(0, 1, n)
    opportunity = r.normal(0, 1, n)
    size = r.normal(0, 1, n)

    # 构造相关误差项
    rho = 0.55
    e1 = r.normal(0, 1, n)
    e2 = r.normal(0, 1, n)
    v = e1
    u = rho * e1 + np.sqrt(1 - rho ** 2) * e2

    # 选择方程：是否获得贷款
    s_star = (
        -0.20
        + 1.00 * collateral
        + 1.05 * bank_access
        - 0.75 * risk
        + 0.20 * opportunity
        + v
    )
    D = (s_star > 0).astype(int)

    # 结果方程：潜在贷款利率
    rate_star = 5.80 - 0.45 * collateral + 0.85 * risk - 0.20 * size + 0.55 * u

    # 只有获得贷款的企业才能观察贷款利率
    loan_rate = np.where(D == 1, rate_star, np.nan)

    return pd.DataFrame({
        "firm_id": np.arange(1, n + 1),
        "collateral": collateral,
        "bank_access": bank_access,
        "risk": risk,
        "opportunity": opportunity,
        "size": size,
        "s_star": s_star,
        "D": D,
        "loan_rate": loan_rate,
        "rate_star": rate_star,
        "u": u,
        "v": v,
    })

df = simulate_heckman_credit()
df.to_csv(DATA_DIR / "heckman_credit_sim.csv", index=False, encoding="utf-8-sig")

df.head()

In [ ]:
# ============================================================
# 2. Heckman two-step 估计
# ============================================================

# Step 1：使用全部样本估计选择方程
Z = sm.add_constant(df[["collateral", "bank_access", "risk"]])
probit = sm.Probit(df["D"], Z).fit(disp=False)

# 根据选择方程计算 inverse Mills ratio
xb = probit.predict(Z, linear=True)
df["imr"] = stats.norm.pdf(xb) / np.maximum(stats.norm.cdf(xb), 1e-12)

# Step 2：只在利率可观测样本中估计结果方程
sel = df["D"] == 1

X_naive = sm.add_constant(df.loc[sel, ["collateral", "risk"]])
naive = sm.OLS(df.loc[sel, "loan_rate"], X_naive).fit()

X_heck = sm.add_constant(df.loc[sel, ["collateral", "risk", "imr"]])
heck = sm.OLS(df.loc[sel, "loan_rate"], X_heck).fit()

summary = pd.DataFrame([
    {
        "model": "Naive OLS",
        "collateral": naive.params["collateral"],
        "risk": naive.params["risk"],
        "imr": np.nan,
    },
    {
        "model": "Heckman two-step",
        "collateral": heck.params["collateral"],
        "risk": heck.params["risk"],
        "imr": heck.params["imr"],
    },
])

summary.to_csv(DATA_DIR / "heckman_two_step_summary.csv", index=False, encoding="utf-8-sig")
summary.round(4)

In [ ]:
# ============================================================
# 3. 图 1：选择性观测机制
# ============================================================

def draw_box(ax, xy, text, fc, ec, w, h=0.82, fontsize=12):
    x, y = xy
    box = FancyBboxPatch(
        (x - w / 2, y - h / 2),
        w,
        h,
        boxstyle="round,pad=0.06,rounding_size=0.08",
        fc=fc,
        ec=ec,
        lw=1.25,
    )
    ax.add_patch(box)
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize)
    return {"x": x, "y": y, "w": w, "h": h}

def edge(box, side):
    if side == "right":
        return (box["x"] + box["w"] / 2, box["y"])
    if side == "left":
        return (box["x"] - box["w"] / 2, box["y"])

def arrow(ax, b1, b2):
    arr = FancyArrowPatch(
        edge(b1, "right"),
        edge(b2, "left"),
        arrowstyle="-|>",
        mutation_scale=12,
        lw=1.15,
        color="0.25",
        shrinkA=8,
        shrinkB=8,
    )
    ax.add_patch(arr)

fig, ax = plt.subplots(figsize=(8.4, 4.6))
ax.axis("off")
ax.set_xlim(0, 8.4)
ax.set_ylim(0, 4.6)

b1 = draw_box(ax, (1.25, 3.2), "抵押能力\ncollateral", "#E8F6F3", "#45B39D", w=1.6)
b2 = draw_box(ax, (1.25, 2.0), "银行可得性\nbank_access", "#FEF9E7", "#D4AC0D", w=1.8)
b3 = draw_box(ax, (1.25, 0.8), "企业风险\nrisk", "#FDEDEC", "#CD6155", w=1.5)

sel_box = draw_box(ax, (4.05, 2.65), "选择方程\n是否获得贷款\nD=1(S*>0)", "#D6EAF8", "#5DADE2", w=2.1, h=1.05)
out_box = draw_box(ax, (4.05, 1.05), "结果方程\n贷款利率 rate\n仅 D=1 可观测", "#EBDEF0", "#AF7AC5", w=2.3, h=1.05)

obs = draw_box(ax, (7.0, 1.85), "观测样本\nrate 不是 0\n而是 missing 或可见", "#D5F5E3", "#58D68D", w=2.2, h=1.08)

for left, right in [
    (b1, sel_box),
    (b1, out_box),
    (b2, sel_box),
    (b3, sel_box),
    (b3, out_box),
    (sel_box, obs),
    (out_box, obs),
]:
    arrow(ax, left, right)

ax.set_title("Heckman selection：贷款利率只在获得贷款样本中可观测", fontsize=14, pad=8)

savefig(fig, "limit_dep_heckman_fig01_selection_mechanism")

In [ ]:
# ============================================================
# 4. 图 2：总体潜在利率与观测利率
# ============================================================

fig, ax = plt.subplots(figsize=(7.6, 4.3))

ax.hist(
    df["rate_star"],
    bins=45,
    alpha=0.45,
    density=True,
    label="潜在贷款利率 (总体)"
)

ax.hist(
    df.loc[df["D"] == 1, "loan_rate"],
    bins=45,
    alpha=0.55,
    density=True,
    label="观测贷款利率 (D=1)"
)

ax.set_xlabel("贷款利率")
ax.set_ylabel("密度")
ax.set_title("只在获得贷款企业中观察利率，会改变样本分布")
ax.legend(frameon=False)

savefig(fig, "limit_dep_heckman_fig02_observed_rate_sample")

In [ ]:
# ============================================================
# 5. 图 3：IMR 校正直觉
# ============================================================

fig, ax = plt.subplots(figsize=(7.6, 4.3))

ax.scatter(
    df.loc[sel, "imr"],
    df.loc[sel, "loan_rate"],
    s=13,
    alpha=0.24
)

coef = np.polyfit(df.loc[sel, "imr"], df.loc[sel, "loan_rate"], 1)
xline = np.linspace(df.loc[sel, "imr"].min(), df.loc[sel, "imr"].max(), 100)

ax.plot(xline, coef[0] * xline + coef[1], lw=2.2)

ax.set_xlabel("inverse Mills ratio")
ax.set_ylabel("观测贷款利率")
ax.set_title("IMR 捕捉被选择样本中的非随机成分")

savefig(fig, "limit_dep_heckman_fig03_imr_correction")

# 运行完成检查

如果上述代码顺利运行，当前目录下应该生成：

- `./data/heckman_credit_sim.csv`
- `./data/heckman_two_step_summary.csv`
- `./figs/limit_dep_heckman_fig01_selection_mechanism.png`
- `./figs/limit_dep_heckman_fig02_observed_rate_sample.png`
- `./figs/limit_dep_heckman_fig03_imr_correction.png`